In [21]:
# pip install langchain_huggingface

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import spacy
import re
import contractions
from textblob import TextBlob
import os
import string

In [23]:
data = TextLoader('rag.txt').load()

In [24]:
data = data[0].page_content

In [25]:
print(data)

What is RAG?
 - It stands for Retrieval Augmented Generation,it is an Ai technique which retieives the relevant data from the external sources 
 and LLM generates Structured output

 - RAG we use for custom chatbots
Example :
Customer Support Chatbots

Why not LLM?
- It is a foundation model which is already trained with vast amount of data to perform text generation tasks.
- It is not having all the information

RAG architecture : 

1) Data loader ?
- Loading the informations from the external sources like pdf,word,text,csv.

2) Text Normalization ?
- Cleaning or removing unwanted data from the document.
- This helps to reduce the tokens and make RAG model to understand the context to generate proper result.

3) Chuncking ?
It converts or splits the document into small parts (chuncks) 

Why Chuncking?
- RAG model can't understand the context if load the entire document(content)
- It helps RAG model to retrieve the actual context based on user query.

4) Embeddings ?
- It is a way of c

In [26]:
var = string.punctuation
data = re.sub(f'[{var}]',' ',data)

In [27]:
data = re.sub(r'\s{2,}','',data).lower()

## Lemmatization

In [28]:
nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
tokens

what is ragit stands for retrieval augmented generation it is an ai technique which retieives the relevant data from the external sourcesand llm generates structured outputrag we use for custom chatbots
examplecustomer support chatbotswhy not llmit is a foundation model which is already trained with vast amount of data to perform text generation tasksit is not having all the informationrag architecture1data loaderloading the informations from the external sources like pdf word text csv2text normalizationcleaning or removing unwanted data from the documentthis helps to reduce the tokens and make rag model to understand the context to generate proper result3chunckingit converts or splits the document into small partschunckswhy chunckingrag model can t understand the context if load the entire document contentit helps rag model to retrieve the actual context based on user query4embeddingsit is a way of converting chuncks into vectors numerical representation of chuncksrag model can t unde

In [29]:
updated_tokens = [str(tokens) for token in tokens if not token.is_stop] # is_stop--> returns boolean if the word is stopword
data = " ".join(updated_tokens)
print(data)
# print(updated_tokens)

what is ragit stands for retrieval augmented generation it is an ai technique which retieives the relevant data from the external sourcesand llm generates structured outputrag we use for custom chatbots
examplecustomer support chatbotswhy not llmit is a foundation model which is already trained with vast amount of data to perform text generation tasksit is not having all the informationrag architecture1data loaderloading the informations from the external sources like pdf word text csv2text normalizationcleaning or removing unwanted data from the documentthis helps to reduce the tokens and make rag model to understand the context to generate proper result3chunckingit converts or splits the document into small partschunckswhy chunckingrag model can t understand the context if load the entire document contentit helps rag model to retrieve the actual context based on user query4embeddingsit is a way of converting chuncks into vectors numerical representation of chuncksrag model can t unde

## **Chuncking**

In [30]:
'''.
Major Types of Chuncking,
- Sentence level Chuncking
- Word level Chuncking
- Fixed size Chuncking
chunck_size = 100
- Chunck Overlap
- Recursive Chuncking
- Semantic Chuncking
'''

'.\nMajor Types of Chuncking,\n- Sentence level Chuncking\n- Word level Chuncking\n- Fixed size Chuncking\nchunck_size = 100\n- Chunck Overlap\n- Recursive Chuncking\n- Semantic Chuncking\n'

In [31]:
# # Fixed Size Chuncking
# s1 = data
# s2 = []
# i = 0
# while i <= len(data):
#     for word in s1:
#         i+=10
#         word = data[:i]
#         s2.append(word)
#         break
# print(s2)

In [32]:
chunk_size = 150
s2 = []
for index in range(0,len(data),chunk_size):
    chunk = data[index:index+chunk_size]
    s2.append(chunk)
print(s2)

['what is ragit stands for retrieval augmented generation it is an ai technique which retieives the relevant data from the external sourcesand llm gener', 'ates structured outputrag we use for custom chatbots\nexamplecustomer support chatbotswhy not llmit is a foundation model which is already trained with', ' vast amount of data to perform text generation tasksit is not having all the informationrag architecture1data loaderloading the informations from the', ' external sources like pdf word text csv2text normalizationcleaning or removing unwanted data from the documentthis helps to reduce the tokens and mak', 'e rag model to understand the context to generate proper result3chunckingit converts or splits the document into small partschunckswhy chunckingrag mo', 'del can t understand the context if load the entire document contentit helps rag model to retrieve the actual context based on user query4embeddingsit', ' is a way of converting chuncks into vectors numerical representation of 

In [34]:
# Chunk Overlap --> It must only 20 to 25% of chunk size 
'''.
chunk_size = 15
chunk_overlap = 7

chunk 1 => what is ragit s
chunk 2 => ragit stands fo
'''
chunk_size = 15
chunk_overlap = 5
s2 = []
for index in range(0,len(data),chunk_size-chunk_overlap):
    chunk = data[index:index+chunk_size]
    s2.append(chunk)
print(s2)

['what is ragit s', 'git stands for ', ' for retrieval ', 'eval augmented ', 'nted generation', 'ation it is an ', 's an ai techniq', 'chnique which r', 'ich retieives t', 'ves the relevan', 'levant data fro', 'a from the exte', ' external sourc', 'sourcesand llm ', ' llm generates ', 'ates structured', 'tured outputrag', 'utrag we use fo', 'se for custom c', 'tom chatbots\nex', 'ts\nexamplecusto', 'customer suppor', 'upport chatbots', 'tbotswhy not ll', 'ot llmit is a f', 's a foundation ', 'tion model whic', ' which is alrea', 'already trained', 'ained with vast', ' vast amount of', 'nt of data to p', ' to perform tex', 'm text generati', 'eration tasksit', 'sksit is not ha', 'ot having all t', 'all the informa', 'formationrag ar', 'ag architecture', 'cture1data load', ' loaderloading ', 'ding the inform', 'nformations fro', 's from the exte', ' external sourc', 'sources like pd', 'ke pdf word tex', 'd text csv2text', '2text normaliza', 'alizationcleani', 'leaning or remo', ' removin